# PCA - Principal Component Analysis
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/03_Machine_Learning/algorithms/pca_dimensionality_reduction.ipynb)

PCA projects data onto orthogonal directions of maximum variance, enabling compression and 2-D visualization of high-dimensional data.

**Covered:** 2-D projection of 64-D digits, explained variance, choosing components, reconstruction quality.

## 1. Digits dataset: 64 dimensions -> 2

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

digits = load_digits()
X = StandardScaler().fit_transform(digits.data)   # PCA is scale-sensitive!

pca2 = PCA(n_components=2)
Z = pca2.fit_transform(X)

plt.figure(figsize=(8, 6))
sc = plt.scatter(Z[:, 0], Z[:, 1], c=digits.target, cmap="tab10", s=12, alpha=0.75)
plt.colorbar(sc, label="digit")
plt.xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.0%} var)")
plt.ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.0%} var)")
plt.title("64-dimensional digits in 2-D"); plt.show()

## 2. How much variance per component?

In [ ]:
pca_full = PCA().fit(X)
cum = pca_full.explained_variance_ratio_.cumsum()

plt.plot(cum, "o-")
plt.axhline(0.95, ls="--", c="r", label="95%")
k95 = (cum < 0.95).sum() + 1
plt.axvline(k95, ls="--", c="g", label=f"{k95} components")
plt.xlabel("#components"); plt.ylabel("cumulative explained variance"); plt.legend()
plt.title(f"{k95}/64 components keep >=95% variance"); plt.show()

## 3. Compression: reconstruct from few components

In [ ]:
import numpy as np
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, k in zip(axes, [4, 10, 20, 32, 64]):
    p = PCA(n_components=k).fit(X)
    img = p.inverse_transform(p.transform(X[[0]]))[0].reshape(8, 8)
    ax.imshow(img, cmap="gray"); ax.axis("off")
    ax.set_title(f"k={k}\n{p.explained_variance_ratio_.sum():.0%} var")
plt.suptitle("Same digit reconstructed from k principal components"); plt.show()

## 4. PCA as noise filter before classification

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
pipe_raw = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), digits.data, digits.target, cv=5)
pipe_pca = cross_val_score(make_pipeline(StandardScaler(), PCA(n_components=0.95), LogisticRegression(max_iter=2000)), digits.data, digits.target, cv=5)
print(f"raw 64-dim : {pipe_raw.mean():.4f}")
print(f"PCA 95% var: {pipe_pca.mean():.4f}  (fewer features, similar/better score)

## Key takeaways
- Always standardize before PCA.
- Use the cumulative-variance curve (`n_components=0.95`) instead of guessing.
- PCA helps visualization, speed, multicollinearity, and mild denoising - but components lose physical interpretability.